# 大模型LoRA微调实战：课程教案与作业整合版

**目标：** 本 Notebook 整合了《阿拉伯语专业领域大模型LoRA微调实战教程》与《2025夏令营大模型微调workshop实践作业》的全部内容。旨在提供一个从数据处理、模型训练、参数调优到多语言迁移的完整、可执行的工作流。

**环境信息：**
- **操作系统：** Linux
- **Python版本：** 3.11.8
- **计算资源：** NVIDIA V100 Tensor Core GPU
- **核心库：** PyTorch 2.5.1, CUDA 12.3

**作业任务分解：**
1.  **基础作业一：** 探索不同LoRA配置（`r`, `lora_alpha`）对模型性能的影响，使用 **ROUGE** 指标进行评估。
2.  **基础作业二：** 实现自定义评估指标（本Notebook中已实现ROUGE指标）。
3.  **进阶作业：** 将微调流程拓展至其他语言（如韩语/俄语），代码中已包含处理该数据集的框架。

## 步骤一：环境安装与设置
首先，我们需要安装所有必需的Python库，并设置Hugging Face的镜像以加速模型下载。

In [ ]:
# 使用清华镜像安装所需依赖包
!pip install transformers==4.41.2 datasets==2.19.0 peft==0.10.0 accelerate==0.30.1 torch==2.5.1 -i https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple
!pip install sentence_transformers scikit-learn rouge_chinese jieba nltk -i https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple

In [ ]:
# 设置Hugging Face国内镜像，加速模型下载
import os
import logging

os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

# 设置日志
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

logger.info(f"Hugging Face Endpoint 设置为: {os.environ.get('HF_ENDPOINT')}")

## 步骤二：数据处理
本部分包含处理原始数据（`.jsonl.gz`格式）的完整流程，将其转换为适合指令微调的格式。

In [ ]:
import json
import gzip
from pathlib import Path
import re
from tqdm import tqdm
import random
from typing import List, Dict

def clean_text(text: str) -> str:
    """清理文本中的多余换行、空格和HTML标签。"""
    if not text:
        return ""
    text = re.sub(r'\n+', '\n', text.strip())
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'<[^>]+>', '', text)
    return text.strip()

def process_item(item: Dict, lang: str = 'ar') -> Dict:
    """处理单条数据，构建统一的指令微调格式。"""
    try:
        if 'title' in item and 'content' in item:
            title = clean_text(item['title'])
            content = clean_text(item['content'])
        elif 'text' in item:
            text = clean_text(item['text'])
            lines = text.split('\n', 1)
            title = lines[0] if len(lines) > 1 else "文章"
            content = lines[1] if len(lines) > 1 else text
        else:
            return None

        if len(content) < 50 or len(content) > 10000:
            return None
        
        lang_map = {
            'ar': '阿拉伯语',
            'ko': '韩语',
            'ru': '俄语'
        }
        lang_name = lang_map.get(lang, '特定语言')

        return {
            "instruction": f"请根据以下标题生成一段带有专业术语的{lang_name}文本。\n\n标题: {title}",
            "input": "",
            "output": content
        }
    except Exception as e:
        logger.error(f"处理数据时出错: {e}")
        return None

def process_raw_file(input_path: str, output_path: str, lang: str = 'ar', sample_ratio: float = 0.25):
    """处理单个原始数据文件并保存。"""
    processed_data = []
    try:
        with gzip.open(input_path, 'rt', encoding='utf-8') as f:
            lines = f.readlines()
            sample_size = max(1, int(len(lines) * sample_ratio))
            sampled_lines = random.sample(lines, sample_size)

            for line in tqdm(sampled_lines, desc=f"处理文件 {Path(input_path).name}"):
                try:
                    item = json.loads(line.strip())
                    processed_item = process_item(item, lang=lang)
                    if processed_item:
                        processed_data.append(processed_item)
                except (json.JSONDecodeError, Exception):
                    continue
    except Exception as e:
        logger.error(f"读取文件 {input_path} 时出错: {e}")
        return

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(processed_data, f, ensure_ascii=False, indent=2)
    
    logger.info(f"处理完成！从 {Path(input_path).name} 中提取了 {len(processed_data)} 条数据。")
    logger.info(f"数据已保存至: {output_path}")

# --- 执行数据处理 ---
random.seed(42)

# 1. 处理阿拉伯语数据 (基础作业)
arabic_raw_path = '/home/mw/input/raw_arabic81628162/阿拉伯part-677f75d865d8-001143.jsonl.gz'
arabic_processed_path = '/home/mw/input/Arabic60526052/lora_training_data_arabic.json' # 教案中使用的路径
if not os.path.exists(arabic_processed_path):
    logger.info("开始处理阿拉伯语原始数据...")
    process_raw_file(arabic_raw_path, arabic_processed_path, lang='ar')
else:
    logger.info(f"已找到处理好的阿拉伯语数据: {arabic_processed_path}")

# 2. 处理韩语/俄语数据 (进阶作业)
# 注意：路径中包含'Russia'，但根据作业上下文可能为韩语数据集，这里我们按俄语处理
korean_raw_path = '/home/mw/input/Russia39613961/part-677f75d865d8-000260.jsonl.gz'
korean_processed_path = '/home/mw/input/Russia39613961/lora_training_data_korean.json'
if not os.path.exists(korean_processed_path):
    logger.info("开始处理韩语/俄语原始数据 (进阶作业)...")
    # 假设该数据集是俄语
    process_raw_file(korean_raw_path, korean_processed_path, lang='ru')
else:
    logger.info(f"已找到处理好的韩语/俄语数据: {korean_processed_path}")

## 步骤三：评估器定义
根据作业要求，我们定义一个评估器，专注于计算ROUGE分数。

In [ ]:
import numpy as np
import torch
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from rouge_chinese import Rouge
import jieba
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

class RougeEvaluator:
    """一个专注于计算ROUGE分数的文本评估器。"""
    def __init__(self):
        self.rouge = Rouge()
    
    def calculate_rouge(self, reference: str, candidate: str) -> Dict[str, float]:
        """计算ROUGE分数"""
        if not candidate or not candidate.strip():
            return {'rouge-1': 0.0, 'rouge-2': 0.0, 'rouge-l': 0.0}
        try:
            reference_cut = ' '.join(jieba.cut(reference))
            candidate_cut = ' '.join(jieba.cut(candidate))
            scores = self.rouge.get_scores(candidate_cut, reference_cut)[0]
            return {
                'rouge-1': scores['rouge-1']['f'],
                'rouge-2': scores['rouge-2']['f'],
                'rouge-l': scores['rouge-l']['f']
            }
        except Exception as e:
            logger.error(f"计算ROUGE分数时出错: {e}")
            return {'rouge-1': 0.0, 'rouge-2': 0.0, 'rouge-l': 0.0}

class TrainingEvaluator:
    """训练过程中的综合评估器"""
    def __init__(self, tokenizer, device):
        self.tokenizer = tokenizer
        self.device = device
        self.rouge_evaluator = RougeEvaluator()
        
        # # --- 以下为教案中原有指标，根据要求注释掉 ---
        # self.sentence_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2').to(device)
        # try:
        #     with open("/home/mw/input/Arabic60526052/domain_terms_arabic.txt", "r", encoding="utf-8") as f:
        #         self.domain_terms = [line.strip() for line in f if line.strip()]
        # except FileNotFoundError:
        #     logger.warning("domain_terms_arabic.txt 未找到，领域指标将受限。")
        #     self.domain_terms = []
        # self.bleu_smooth = SmoothingFunction()

    def evaluate_generation(self, model, validation_prompts: List[Dict]) -> Dict[str, float]:
        """评估生成文本的ROUGE分数"""
        model.eval()
        all_rouge_scores = {'rouge-1': [], 'rouge-2': [], 'rouge-l': []}
        
        with torch.no_grad():
            for item in tqdm(validation_prompts, desc="Evaluating Generation with ROUGE"):
                prompt = item['instruction']
                reference_output = item['output']

                inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(self.device)
                outputs = model.generate(
                    **inputs,
                    max_length=1024, # 增加生成长度以获得更完整的回答
                    num_return_sequences=1,
                    do_sample=True,
                    temperature=0.7,
                    top_p=0.95,
                    pad_token_id=self.tokenizer.eos_token_id
                )
                response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
                generated_output = response.split("### Response:")[-1].strip()

                # 计算ROUGE分数
                rouge_scores = self.rouge_evaluator.calculate_rouge(reference_output, generated_output)
                for key, value in rouge_scores.items():
                    all_rouge_scores[key].append(value)

        avg_metrics = {f"avg_{k}": np.mean(v) for k, v in all_rouge_scores.items()}
        return avg_metrics

## 步骤四：模型训练与参数搜索 (基础作业一)
这是课程的核心部分。我们将：
1. 定义一个数据加载函数。
2. 循环遍历不同的LoRA超参数（`r` 和 `lora_alpha`）。
3. 对每组参数，进行短暂的训练。
4. 使用ROUGE分数评估模型性能。
5. 找出并保存性能最佳的模型。

In [ ]:
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType
from torch.optim import AdamW

def load_and_prepare_data(json_path, tokenizer, max_length=512, val_size=0.1):
    """加载、处理并划分指令微调数据。"""
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    formatted_texts = []
    for item in data:
        if 'output' not in item: item['output'] = ''
        text = f"### Instruction:\n{item['instruction']}\n\n### Response:\n{item['output']}"
        formatted_texts.append(text)

    encodings = tokenizer(formatted_texts, truncation=True, max_length=max_length, padding="max_length")
    dataset = Dataset.from_dict(encodings)
    dataset = dataset.map(lambda examples: {'labels': examples['input_ids'].copy()}, batched=True)

    split_dataset = dataset.train_test_split(test_size=val_size, seed=42)
    train_dataset = split_dataset['train']
    val_dataset = split_dataset['test']
    
    val_indices = split_dataset['test']._indices
    validation_prompts = [data[i] for i in val_indices]

    return train_dataset, val_dataset, validation_prompts

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def main_training_loop():
    set_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_path = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
    data_path = "/home/mw/input/Arabic60526052/lora_training_data_arabic.json"

    logger.info(f"设备: {device}, 模型: {model_path}")

    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    train_dataset, val_dataset, validation_prompts = load_and_prepare_data(data_path, tokenizer)
    train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True) # V100可以尝试更大的batch_size
    # val_dataloader = DataLoader(val_dataset, batch_size=4)

    lora_configs_to_try = [
        {"r": 8, "lora_alpha": 16, "lora_dropout": 0.05},
        {"r": 16, "lora_alpha": 32, "lora_dropout": 0.1},
        {"r": 32, "lora_alpha": 64, "lora_dropout": 0.1}
    ]

    best_rouge_l = -1
    best_config = None
    all_results = []

    for config in lora_configs_to_try:
        logger.info(f"--- 开始测试配置: r={config['r']}, alpha={config['lora_alpha']} ---")
        
        base_model = AutoModelForCausalLM.from_pretrained(
            model_path, trust_remote_code=True, torch_dtype=torch.bfloat16 # V100支持bfloat16
        ).to(device)

        lora_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM, **config,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], bias="none"
        )
        peft_model = get_peft_model(base_model, lora_config)
        peft_model.print_trainable_parameters()

        optimizer = AdamW(peft_model.parameters(), lr=5e-5)
        num_epochs = 1 # 为快速迭代，每个配置只训练1个epoch

        for epoch in range(num_epochs):
            peft_model.train()
            for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1} with r={config['r']}"):
                batch = {k: v.to(device) for k, v in batch.items() if k in ['input_ids', 'attention_mask', 'labels']}
                outputs = peft_model(**batch)
                loss = outputs.loss
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()

        logger.info("开始评估...")
        evaluator = TrainingEvaluator(tokenizer, device)
        rouge_metrics = evaluator.evaluate_generation(peft_model, validation_prompts[:20]) # 取20个样本快速评估
        
        logger.info(f"配置 r={config['r']} 的评估结果:")
        for key, value in rouge_metrics.items():
            logger.info(f"  - {key}: {value:.4f}")
        
        current_results = {"config": config, **rouge_metrics}
        all_results.append(current_results)

        if rouge_metrics['avg_rouge-l'] > best_rouge_l:
            best_rouge_l = rouge_metrics['avg_rouge-l']
            best_config = config
            logger.info(f"*** 发现新的最佳配置: {config} ***")
            # 使用教案中的路径保存最佳模型
            best_model_path = "/home/mw/input/Arabic314891489/deepseek-lora-best"
            peft_model.save_pretrained(best_model_path)
            tokenizer.save_pretrained(best_model_path)
            logger.info(f"最佳模型已保存至: {best_model_path}")

    logger.info("\n--- 所有配置测试完成 ---")
    for result in all_results:
        logger.info(f"配置: {result['config']} -> ROUGE-L: {result['avg_rouge-l']:.4f}")
    logger.info(f"\n最终最佳配置为: {best_config}")

# 执行训练和参数搜索
main_training_loop()

## 步骤五：模型合并与测试
训练完成后，我们将性能最佳的LoRA权重与基础模型合并，形成一个完整的、可以直接部署的模型，并进行最终的生成效果测试。

In [ ]:
from peft import PeftModel

def merge_and_test():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    base_model_path = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
    lora_adapter_path = "/home/mw/input/Arabic314891489/deepseek-lora-best"
    merged_model_path = "/home/mw/input/Arabic314891489/deepseek-merged-final"

    logger.info("开始加载基础模型用于合并...")
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_path,
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
        device_map="auto"
    )

    logger.info(f"加载LoRA适配器从: {lora_adapter_path}")
    model_to_merge = PeftModel.from_pretrained(base_model, lora_adapter_path)

    logger.info("开始合并权重...")
    merged_model = model_to_merge.merge_and_unload()
    logger.info("权重合并完成。")

    logger.info(f"保存合并后的模型至: {merged_model_path}")
    merged_model.save_pretrained(merged_model_path)
    tokenizer = AutoTokenizer.from_pretrained(lora_adapter_path)
    tokenizer.save_pretrained(merged_model_path)
    logger.info("模型和分词器已保存。")

    # --- 测试合并后的模型 ---
    logger.info("\n--- 测试合并后的模型生成效果 ---")
    test_prompts = [
        "请解释阿拉伯语中'السوق'这个术语的含义。",
        "ما معنى مصطلح 'الذكاء الاصطناعي'؟", # "人工智能"这个术语是什么意思？
        "اكتب فقرة قصيرة عن أهمية الطاقة المتجددة.", # 写一段关于可再生能源重要性的短文
    ]
    
    for prompt in test_prompts:
        logger.info(f"\n测试提示: {prompt}")
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        
        with torch.no_grad():
            outputs = merged_model.generate(
                **inputs,
                max_length=256,
                num_return_sequences=1,
                do_sample=True,
                temperature=0.7,
                top_p=0.95,
                pad_token_id=tokenizer.eos_token_id
            )
        
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        logger.info(f"模型回答:\n{response}")
        logger.info("-" * 50)

# 执行模型合并与测试
merge_and_test()

## 步骤六：总结与进阶作业展望

至此，我们完成了基础作业的核心部分：
1.  **数据处理**：成功将原始的阿拉伯语料处理成指令微调格式。
2.  **参数搜索**：通过循环测试，找到了在ROUGE-L指标上表现最佳的LoRA超参数组合。
3.  **模型训练与保存**：训练并保存了性能最佳的LoRA模型。
4.  **模型合并**：将LoRA权重与基础模型合并，得到了一个独立的、性能增强的模型。

### 进阶作业指导
要完成进阶作业（微调韩语/俄语模型），您需要：
1.  **修改数据路径**：在`main_training_loop`函数中，将`data_path`变量指向处理好的韩语/俄语数据文件路径（例如 `/home/mw/input/Russia39613961/lora_training_data_korean.json`）。
2.  **实现特定语言分词（可选但推荐）**：为了更精确地计算ROUGE分数，您可以在`RougeEvaluator`中为目标语言（如韩语）实现特定的分词逻辑，而不是统一使用`jieba`。
3.  **重新运行训练流程**：执行修改后的Notebook，即可开始对新语言模型的微调和评估。